In [11]:
import os
import tempfile
from pathlib import Path

from keras.src.layers.merging import concatenate

# custom imports
from src.utils.data_fetch import DataLoader
from src.utils.model_eval.evaluation import evaluate_predictions
from src.visualisations.viz_model import plot_confusion_matrix

# for implementing neural network
import tensorflow as tf
from tensorflow import keras
import random
import re

# other essential libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# for data preprocessing
from sklearn.preprocessing import (
    MinMaxScaler, FunctionTransformer,
    OneHotEncoder, OrdinalEncoder, LabelEncoder
)
from sklearn.pipeline import Pipeline

# various scoring metrics
from sklearn.metrics import (classification_report, accuracy_score,
                             recall_score, precision_score,
                             f1_score, roc_auc_score)

# for hyperparameter tuning
import optuna
from functools import partial


def find_project_root(start=None):
    current = Path.cwd() if start is None else Path(start).resolve()
    for path in [current, *current.parents]:
        if (path / '.paths').exists() or (path / 'pyproject.toml').exists():
            return path
    return current


# export path for the final model:
project_root = find_project_root()
export_dir = project_root / 'models' / 'dnn_v2_sel_feats'
export_dir.mkdir(parents=True, exist_ok=True)

SEED = 1234


####  1.2 Loading the data

In [2]:
from src.utils.data_fetch import import_data

X_train, X_val, X_test, y_train, y_val, y_test = import_data(version='final_raw',
                                                            access='remote')

Data loaded from path:  https://raw.githubusercontent.com/wip-0/ds207_final_project/refs/heads/main/data/final_processed/base_kf/raw/X_train_mini.csv 
Data loaded from path:  https://raw.githubusercontent.com/wip-0/ds207_final_project/refs/heads/main/data/final_processed/base_kf/raw/X_val.csv 
Data loaded from path:  https://raw.githubusercontent.com/wip-0/ds207_final_project/refs/heads/main/data/final_processed/base_kf/raw/X_test.csv 
Data loaded from path:  https://raw.githubusercontent.com/wip-0/ds207_final_project/refs/heads/main/data/final_processed/base_kf/raw/y_train_mini.csv 
Data loaded from path:  https://raw.githubusercontent.com/wip-0/ds207_final_project/refs/heads/main/data/final_processed/base_kf/raw/y_val.csv 
Data loaded from path:  https://raw.githubusercontent.com/wip-0/ds207_final_project/refs/heads/main/data/final_processed/base_kf/raw/y_test.csv 
Data imported succesfully from paths


#### 1.3 Preprocessing Numeric and Categorical Data

In [4]:
categorical_features = [
    'pipeline_raw__race',
    'pipeline_raw__age',
    'pipeline_admission_type_raw__admission_type_id',
    'pipeline_discharge_raw__discharge_disposition_id',
    'pipeline_medical_raw__medical_specialty'
]

numeric_features = [
    'time_in_hospital',
    'num_lab_procedures',
    'num_procedures',
    'num_medications',
    'number_outpatient',
    'number_emergency',
    'number_inpatient',
    'number_diagnoses'
]

zero_heavy_numeric_features = [
    'num_procedures',
    'number_outpatient',
    'number_emergency',
    'number_inpatient'
]

**Numeric features are transformed with train-only fitting.**

**Important**: Zero-heavy count features also get binary indicators.

In [5]:
def numeric_features_pipeline(X_train_raw, X_val_raw, X_test_raw,
                              numeric_cols= numeric_features,
                              zero_heavy_cols= zero_heavy_numeric_features):
    #### PIPELINE for numeric features ####
    # use log1p because some columns contain zeros:
    # log1p(x) = log(1 + x)
    log_minmax_pipeline = Pipeline(steps=[
        ('log', FunctionTransformer(np.log1p, feature_names_out='one-to-one')),
        ('minmax_scaler', MinMaxScaler(feature_range=(0, 1)))
    ])

    X_train_num = log_minmax_pipeline.fit_transform(X_train_raw[numeric_cols])
    X_val_num = log_minmax_pipeline.transform(X_val_raw[numeric_cols])
    X_test_num = log_minmax_pipeline.transform(X_test_raw[numeric_cols])

    X_train_num = pd.DataFrame(X_train_num, columns=numeric_cols, index=X_train_raw.index)
    X_val_num = pd.DataFrame(X_val_num, columns=numeric_cols, index=X_val_raw.index)
    X_test_num = pd.DataFrame(X_test_num, columns=numeric_cols, index=X_test_raw.index)

    # add zero-vs-nonzero indicators for count features with many zeros.
    indicator_cols = []
    for col in zero_heavy_cols:
        indicator_col = f'had_{col.replace("number_", "")}'
        indicator_cols.append(indicator_col)
        X_train_num[indicator_col] = (X_train_raw[col] > 0).astype('float32')
        X_val_num[indicator_col] = (X_val_raw[col] > 0).astype('float32')
        X_test_num[indicator_col] = (X_test_raw[col] > 0).astype('float32')

    numeric_model_features = numeric_cols + indicator_cols

    return X_train_num, X_val_num, X_test_num, numeric_model_features, log_minmax_pipeline

**Transforming categorical features with Lable/Ordinal encoding**

In [15]:
def categorical_features_pipeline(
    X_train_raw,
    X_val_raw,
    X_test_raw,
    categorical_cols= categorical_features,
    encoding_map=None,
    ordinal_categories=None,
    default_encoding='onehot',
    missing_value='Missing'
):
    """
    Pipeline for categorical features.
    """

    if encoding_map is None:
        encoding_map = {}

    if ordinal_categories is None:
        ordinal_categories = {}

    allowed_encodings = {'ordinal', 'label'}

    # Decide which encoding each categorical column should use.
    col_encoding = {
        col: encoding_map.get(col, default_encoding)
        for col in categorical_cols
    }

    invalid_encodings = set(col_encoding.values()) - allowed_encodings
    if invalid_encodings:
        raise ValueError(f'Invalid encodings found: {invalid_encodings}')

    # Work on copies and replace missing values consistently.
    X_train_cat_raw = X_train_raw[categorical_cols].copy().fillna(missing_value).astype(str)
    X_val_cat_raw = X_val_raw[categorical_cols].copy().fillna(missing_value).astype(str)
    X_test_cat_raw = X_test_raw[categorical_cols].copy().fillna(missing_value).astype(str)

    transformed_train_parts = []
    transformed_val_parts = []
    transformed_test_parts = []

    categorical_model_features = []
    categorical_preprocessors = {}

    ordinal_cols = [col for col, enc in col_encoding.items() if enc == 'ordinal']
    label_cols = [col for col, enc in col_encoding.items() if enc == 'label']
    onehot_cols = [col for col, enc in col_encoding.items() if enc == 'onehot']

    #### Ordinal Encoding ####
    # Use this when categories have a meaningful order, e.g. age ranges.
    for col in ordinal_cols:
        categories = ordinal_categories.get(col, 'auto')

        encoder = OrdinalEncoder(
            categories=[categories] if categories != 'auto' else 'auto',
            handle_unknown='use_encoded_value',
            unknown_value=-1
        )

        X_train_encoded = encoder.fit_transform(X_train_cat_raw[[col]])
        X_val_encoded = encoder.transform(X_val_cat_raw[[col]])
        X_test_encoded = encoder.transform(X_test_cat_raw[[col]])

        transformed_train_parts.append(
            pd.DataFrame(X_train_encoded, columns=[col],
                         index=X_train_raw.index)
        )
        transformed_val_parts.append(
            pd.DataFrame(X_val_encoded, columns=[col],
                         index=X_val_raw.index)
        )
        transformed_test_parts.append(
            pd.DataFrame(X_test_encoded, columns=[col],
                         index=X_test_raw.index)
        )

        categorical_model_features.append(col)
        categorical_preprocessors[col] = encoder

    #### Label Encoding ####
    # LabelEncoder does not naturally handle unseen categories, so unseen
    # validation/test categories are mapped to -1.
    for col in label_cols:
        encoder = LabelEncoder()
        encoder.fit(X_train_cat_raw[col])

        class_mapping = {
            label: idx
            for idx, label in enumerate(encoder.classes_)
        }

        X_train_encoded = X_train_cat_raw[col].map(class_mapping).fillna(-1).astype('float32')
        X_val_encoded = X_val_cat_raw[col].map(class_mapping).fillna(-1).astype('float32')
        X_test_encoded = X_test_cat_raw[col].map(class_mapping).fillna(-1).astype('float32')

        transformed_train_parts.append(
            pd.DataFrame({col: X_train_encoded},
                         index= X_train_raw.index)
        )
        transformed_val_parts.append(
            pd.DataFrame({col: X_val_encoded},
                         index= X_val_raw.index)
        )
        transformed_test_parts.append(
            pd.DataFrame({col: X_test_encoded},
                         index= X_test_raw.index)
        )

        categorical_model_features.append(col)
        categorical_preprocessors[col] = {
            'encoder': encoder,
            'class_mapping': class_mapping
        }

    # concatenate all transformed categorical features
    # into one dataframe
    X_train_cat = pd.concat(transformed_train_parts, axis= 1)
    X_val_cat = pd.concat(transformed_val_parts, axis= 1)
    X_test_cat = pd.concat(transformed_test_parts, axis= 1)

    return (
        X_train_cat,
        X_val_cat,
        X_test_cat,
        categorical_model_features,
        categorical_preprocessors
    )

In [ ]:
def concatenate_features(df_numeric, df_cat):
 return pd.concat([df_numeric, df_cat], axis=1)

**Applying the numeric features pipeline:**

In [12]:
# creating a copy of the initial data:
X_train_raw = X_train.copy()
X_val_raw = X_val.copy()
X_test_raw = X_test.copy()

In [13]:
# transforming numeric features:
X_train_num, X_val_num, X_test_num, numeric_model_features, numeric_preprocessor = \
numeric_features_pipeline(X_train_raw, X_val_raw, X_test_raw)

X_train_num.head()

,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,number_diagnoses,had_num_procedures,had_outpatient,had_emergency,had_inpatient
0,0.344010,0.810349,0.000000,0.606234,0.000000,0.0,0.000000,0.752051,0.0,0.0,0.0,0.0
1,0.201233,0.426894,0.920782,0.524000,0.292091,0.0,0.224244,0.585385,1.0,1.0,0.0,1.0
2,0.201233,0.741807,0.356207,0.576282,0.000000,0.0,0.000000,0.647781,1.0,0.0,0.0,0.0
3,0.000000,0.776254,0.000000,0.405022,0.000000,0.0,0.000000,0.513354,0.0,0.0,0.0,0.0
4,0.344010,0.660580,1.000000,0.576282,0.000000,0.0,0.000000,0.752051,1.0,0.0,0.0,0.0


**Applying the categorical features pipeline:**

In [16]:
categorical_encoding_map = {
    'pipeline_raw__race': 'label',
    'pipeline_raw__age': 'ordinal',
    'pipeline_admission_type_raw__admission_type_id': 'label',
    'pipeline_discharge_raw__discharge_disposition_id': 'label',
    'pipeline_medical_raw__medical_specialty': 'label'
}

ordinal_categories = {
    'pipeline_raw__age': [
        '[0-10)', '[10-20)', '[20-30)', '[30-40)', '[40-50)',
        '[50-60)', '[60-70)', '[70-80)', '[80-90)', '[90-100)'
    ]
}

X_train_cat, X_val_cat, X_test_cat, categorical_model_features, categorical_preprocessor = \
categorical_features_pipeline(X_train_raw, X_val_raw, X_test_raw,
                                encoding_map= categorical_encoding_map,
                                    ordinal_categories= ordinal_categories)

X_train_cat.head()

,pipeline_raw__age,pipeline_raw__race,pipeline_admission_type_raw__admission_type_id,pipeline_discharge_raw__discharge_disposition_id,pipeline_medical_raw__medical_specialty
0,1.0,2.0,1.0,0.0,11.0
1,2.0,0.0,1.0,0.0,11.0
2,3.0,2.0,1.0,0.0,11.0
3,4.0,2.0,1.0,0.0,11.0
4,5.0,2.0,1.0,0.0,11.0


In [17]:
X_train = pd.concat([X_train_num, X_train_cat], axis=1)
X_val = pd.concat([X_val_num, X_val_cat], axis=1)
X_test = pd.concat([X_test_num, X_test_cat], axis=1)

X_train.columns

Index(['time_in_hospital', 'num_lab_procedures', 'num_procedures',
       'num_medications', 'number_outpatient', 'number_emergency',
       'number_inpatient', 'number_diagnoses', 'had_num_procedures',
       'had_outpatient', 'had_emergency', 'had_inpatient', 'pipeline_raw__age',
       'pipeline_raw__race', 'pipeline_admission_type_raw__admission_type_id',
       'pipeline_discharge_raw__discharge_disposition_id',
       'pipeline_medical_raw__medical_specialty'],
      dtype='str')

### Creating the ANFIS model:

In [ ]:
from skanfis import scikit_anfis

In [19]:
from skanfis import scikit_anfis
from sklearn.metrics import accuracy_score

# Convert features and labels to NumPy arrays.
# ANFIS expects numeric inputs only.
X_train_anfis = X_train.to_numpy(dtype=np.float32)
X_val_anfis = X_val.to_numpy(dtype=np.float32)
X_test_anfis = X_test.to_numpy(dtype=np.float32)

y_train_anfis = y_train.squeeze().to_numpy(dtype=np.float32)
y_val_anfis = y_val.squeeze().to_numpy(dtype=np.float32)
y_test_anfis = y_test.squeeze().to_numpy(dtype=np.float32)

In [ ]:
# Create a Scikit-ANFIS fuzzy system object
# label="c" tells
# the package this is a classification problem.
model_anfis = scikit_anfis(
    data= X_train_anfis,
     description= "Readmitted_Patients",
      epoch= 100,
        hybrid= True,
         label= "c",
          show_banner= True
)

# Train model.
model_anfis.fit(X_train_anfis, y_train_anfis)

In [ ]:
# Predict validation labels.
y_pred = model_anfis.predict(X_val_anfis).ravel().astype(int)

In [ ]:
print("Model Accuracy: ", accuracy_score(y_pred, y_val))